# FinQwen: Finance-Tuned LLM Benchmark
**Model**: Qwen2.5-7B-Instruct (4-bit) | **Framework**: Unsloth + LoRA | **Domain**: Financial NLP

Phases:
1. Install & Setup
2. Load Model + LoRA
3. Dataset Preparation + EDA
4. Training
5. Evaluation (6 metrics)
6. Results Table
7. Save + Export (GGUF / HuggingFace)


## Phase 1 — Install Dependencies

In [ ]:
# Install unsloth first (must be before transformers to avoid conflicts)
!pip install unsloth -q
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git -q

# Core training
!pip install trl transformers accelerate peft bitsandbytes datasets -q

# Evaluation
!pip install evaluate rouge_score scikit-learn groq -q

# Visualization
!pip install plotly pandas matplotlib -q

print("All dependencies installed.")

## Phase 2 — Load Model + Apply LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # auto-detect bf16/fp16
    load_in_4bit=True,
)

print(f"Model loaded. Device: {next(model.parameters()).device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
# Apply LoRA with aggressive settings (not toy r=4)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                  # rank — meaningful adaptation
    lora_alpha=32,         # alpha = 2x rank is standard
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention
        "gate_proj", "up_proj", "down_proj",       # MLP — important for domain knowledge
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=True,       # rank-stabilized LoRA — free performance gain
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable/1e6:.2f}M / {total/1e6:.0f}M ({100*trainable/total:.2f}%)")

## Phase 3 — Dataset Preparation + EDA

In [ ]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import matplotlib.pyplot as plt

# Load both finance datasets
print("Loading datasets...")

configs = ['ConvFinQA', 'FiQA_SA', 'FPB', 'Headline', 'NER']
all_finance_tasks = []
for config_name in configs:
    print(f"Loading AdaptLLM/finance-tasks with config: {config_name}...")
    all_finance_tasks.append(load_dataset("AdaptLLM/finance-tasks", config_name, split="test")) # Changed split to 'test'
finance_tasks = concatenate_datasets(all_finance_tasks)

fingpt_sentiment = load_dataset("FinGPT/fingpt-sentiment-train", split="train")

print(f"finance-tasks: {len(finance_tasks)} rows | columns: {finance_tasks.column_names}")
print(f"fingpt-sentiment: {len(fingpt_sentiment)} rows | columns: {fingpt_sentiment.column_names}")

In [ ]:
# EDA — token length distribution
sample_texts = [finance_tasks[i].get('input', '') + finance_tasks[i].get('output', '')
                for i in range(min(500, len(finance_tasks)))]
token_lengths = [len(tokenizer.encode(t)) for t in sample_texts]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(token_lengths, bins=40, color='steelblue', edgecolor='white')
plt.xlabel('Token length')
plt.ylabel('Count')
plt.title('Token length distribution (finance-tasks sample)')
plt.axvline(MAX_SEQ_LENGTH, color='red', linestyle='--', label=f'max={MAX_SEQ_LENGTH}')
plt.legend()

# FinGPT label distribution
plt.subplot(1, 2, 2)
if 'output' in fingpt_sentiment.column_names:
    labels = pd.Series(fingpt_sentiment['output']).value_counts()
    labels.plot(kind='bar', color=['green', 'gray', 'red'])
    plt.title('Sentiment label distribution')
    plt.xlabel('Label')
    plt.ylabel('Count')
    plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig('eda_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Samples > {MAX_SEQ_LENGTH} tokens: {sum(l > MAX_SEQ_LENGTH for l in token_lengths)} / {len(token_lengths)}")

In [ ]:
# Format datasets into ShareGPT format (works with Qwen chat template)
def format_finance_tasks(example):
    instruction = example.get('input', '')
    response = example.get('output', '')
    return {
        "conversations": [
            {"from": "human", "value": instruction},
            {"from": "gpt", "value": response}
        ]
    }

def format_fingpt(example):
    instruction = example.get('input', '')
    response = example.get('output', '')
    return {
        "conversations": [
            {"from": "human", "value": f"Analyze the financial sentiment of the following text:\n{instruction}"},
            {"from": "gpt", "value": response}
        ]
    }

finance_formatted = finance_tasks.map(format_finance_tasks, remove_columns=finance_tasks.column_names)
fingpt_formatted  = fingpt_sentiment.map(format_fingpt, remove_columns=fingpt_sentiment.column_names)

# Subsample to balance (avoid sentiment dominating)
MAX_TRAIN = 8000
finance_sub = finance_formatted.select(range(min(MAX_TRAIN // 2, len(finance_formatted))))
fingpt_sub  = fingpt_formatted.select(range(min(MAX_TRAIN // 2, len(fingpt_formatted))))

combined = concatenate_datasets([finance_sub, fingpt_sub]).shuffle(seed=42)
print(f"Combined training set: {len(combined)} examples")

In [ ]:
# Apply Qwen chat template
from unsloth import standardize_sharegpt, apply_chat_template

combined = standardize_sharegpt(combined)

chat_template = """<|im_start|>system
You are FinQwen, an expert financial analysis assistant. Answer accurately using financial domain knowledge.
<|im_end|>
<|im_start|>user
{INPUT}<|im_end|>
<|im_start|>assistant
{OUTPUT}<|im_end|>"""

dataset = apply_chat_template(
    combined,
    tokenizer=tokenizer,
    chat_template=chat_template,
)

print("Sample formatted:")
print(dataset[0]['text'][:500])

## Phase 4 — Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,       # effective batch = 8
        warmup_steps=20,
        max_steps=300,                       # real training, not demo
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        save_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",          # cosine beats linear for domain fine-tuning
        seed=42,
        output_dir="finqwen_checkpoints",
        report_to="none",                    # switch to 'wandb' if you want tracking
    ),
)

print("Trainer configured. Starting training...")
trainer_stats = trainer.train()

print(f"\nTraining complete.")
print(f"Total steps: {trainer_stats.global_step}")
print(f"Final loss: {trainer_stats.training_loss:.4f}")
print(f"Time: {trainer_stats.metrics['train_runtime']/60:.1f} min")

In [ ]:
# Plot loss curve (save for LinkedIn post)
import json
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [x['step'] for x in log_history if 'loss' in x]
losses = [x['loss'] for x in log_history if 'loss' in x]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, color='steelblue', linewidth=2)
plt.xlabel('Step')
plt.ylabel('Training Loss')
plt.title('FinQwen Training Loss Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print("Saved: loss_curve.png")

## Phase 5 — Evaluation (6 Metrics)

In [ ]:
# Setup: load base model for comparison
# NOTE: set your Groq API key below
import os
from groq import Groq

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"  # get from console.groq.com  # <-- paste your key here

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Switch fine-tuned model to inference mode
FastLanguageModel.for_inference(model)

def generate_finetuned(prompt, max_new_tokens=256):
    """Generate from the fine-tuned FinQwen model."""
    messages = [
        # Remove the system message here as it's already part of the chat template.
        # {"role": "system", "content": "You are FinQwen, an expert financial analysis assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def generate_groq(prompt, model_name="llama-3.1-8b-instant", max_tokens=256):
    """Generate from Groq (used as base model proxy + judge)."""
    resp = groq_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.1,
    )
    return resp.choices[0].message.content

print("Inference functions ready.")

In [ ]:
# 50-question benchmark — finance domain
BENCHMARK_QA = [
    # Sentiment classification (10)
    {"type": "sentiment", "input": "Tesla reported record quarterly deliveries, beating analyst expectations by 15%.", "label": "positive"},
    {"type": "sentiment", "input": "The company announced significant write-downs and missed revenue targets for the third consecutive quarter.", "label": "negative"},
    {"type": "sentiment", "input": "The Federal Reserve held interest rates steady, in line with market expectations.", "label": "neutral"},
    {"type": "sentiment", "input": "Apple's Services revenue grew 16% year-over-year, driven by strong App Store and subscription performance.", "label": "positive"},
    {"type": "sentiment", "input": "The bank disclosed $2.3B in potential losses tied to commercial real estate exposure.", "label": "negative"},
    {"type": "sentiment", "input": "The merger was approved by shareholders pending regulatory review.", "label": "neutral"},
    {"type": "sentiment", "input": "NVIDIA's data center segment surpassed $10B in quarterly revenue for the first time.", "label": "positive"},
    {"type": "sentiment", "input": "The company is undergoing a strategic review amid declining market share.", "label": "negative"},
    {"type": "sentiment", "input": "Management maintained full-year guidance without revision.", "label": "neutral"},
    {"type": "sentiment", "input": "Earnings per share exceeded consensus by 22%, driven by cost discipline and margin expansion.", "label": "positive"},
    # Summarization (10)
    {"type": "summarization", "input": "Summarize the key financial metrics from: Revenue grew 12% YoY to $45.2B. Gross margin improved 200bps to 42%. Operating income was $8.1B, up 18% YoY. Free cash flow reached $6.3B. The company returned $3B to shareholders via buybacks. EPS came in at $2.41 vs $2.10 expected.", "reference": "Revenue rose 12% to $45.2B, gross margin improved to 42%, operating income grew 18% to $8.1B, FCF was $6.3B, and EPS of $2.41 beat estimates by 15%."},
    {"type": "summarization", "input": "Summarize: The Federal Reserve raised interest rates by 25 basis points to a target range of 5.25%-5.50%, the highest level in 22 years. Chair Powell signaled that future rate decisions would be data-dependent, neither committing to additional hikes nor to cuts in the near term.", "reference": "The Fed raised rates 25bps to 5.25-5.50%, a 22-year high, with future moves to be data-driven."},
    {"type": "summarization", "input": "Summarize: Amazon Web Services grew 17% YoY to $24.2B, contributing 67% of Amazon's total operating income. Management attributed growth to AI workload adoption and new enterprise contracts signed during the quarter.", "reference": "AWS grew 17% to $24.2B, driving 67% of Amazon's operating income, boosted by AI and new enterprise wins."},
    {"type": "summarization", "input": "Summarize: The company recorded a $1.2B goodwill impairment charge related to its 2021 acquisition. Adjusted EBITDA margin compressed 400bps to 18%. Management lowered full-year guidance by 8% citing macroeconomic headwinds.", "reference": "A $1.2B goodwill impairment hit results, EBITDA margin fell 400bps to 18%, and guidance was cut 8% due to macro pressures."},
    {"type": "summarization", "input": "Summarize: Visa reported payment volume growth of 8% globally, with cross-border transactions rising 16% YoY. Net revenue increased 10% to $8.9B. The company announced a $2.5B accelerated share repurchase program.", "reference": "Visa saw 8% payment volume growth, 16% cross-border gains, revenue up 10% to $8.9B, with a $2.5B buyback announced."},
    {"type": "summarization", "input": "Summarize: JPMorgan Chase reported net income of $13.1B, up 35% YoY. Net interest income of $22.9B benefited from higher rates. Provision for credit losses increased to $2.9B. The investment banking fee pool contracted 15%.", "reference": "JPMorgan earned $13.1B net income (+35%), with $22.9B NII on higher rates, $2.9B credit loss provision, and IB fees down 15%."},
    {"type": "summarization", "input": "Summarize: Microsoft's Intelligent Cloud segment grew 21% to $28.5B. Azure revenue grew 29%, ahead of the 26% consensus estimate. AI services contributed approximately 7 percentage points to Azure's growth.", "reference": "Microsoft's Intelligent Cloud hit $28.5B (+21%), Azure grew 29% (beating 26% consensus), with AI adding 7pp to growth."},
    {"type": "summarization", "input": "Summarize: The IPO priced at the top of its range at $18 per share, raising $540M. The company is valued at $4.2B. Shares surged 32% on the first day of trading. The offering was 8x oversubscribed.", "reference": "IPO priced at $18, raised $540M at a $4.2B valuation, surged 32% on debut, and was 8x oversubscribed."},
    {"type": "summarization", "input": "Summarize: The hedge fund disclosed a 9.8% passive stake in the company. The filing indicated no plans to seek board representation or advocate for strategic changes. The position was acquired over six months.", "reference": "A hedge fund disclosed a 9.8% passive stake acquired over six months, with no plans for board seats or strategic changes."},
    {"type": "summarization", "input": "Summarize: Gross merchandise value declined 4% to $31.2B. Take rate improved 40bps to 14.2%. Adjusted EBITDA loss narrowed by $200M to -$150M. Management expects profitability by Q4.", "reference": "GMV fell 4% to $31.2B, take rate improved to 14.2%, EBITDA loss narrowed to -$150M, with profitability expected in Q4."},
    # Financial QA — factual knowledge (15)
    {"type": "qa", "input": "What is the formula for Return on Equity (ROE)?", "reference": "ROE = Net Income / Shareholders' Equity"},
    {"type": "qa", "input": "What does a P/E ratio measure?", "reference": "Price-to-earnings ratio measures how much investors pay per dollar of earnings."},
    {"type": "qa", "input": "What is free cash flow and why is it important?", "reference": "FCF = Operating Cash Flow - Capital Expenditures. It represents cash available after maintaining/expanding the asset base."},
    {"type": "qa", "input": "Explain the difference between gross profit and operating profit.", "reference": "Gross profit = Revenue - COGS. Operating profit = Gross profit - Operating expenses (SG&A, R&D)."},
    {"type": "qa", "input": "What is EBITDA and what are its limitations?", "reference": "EBITDA = Earnings Before Interest, Taxes, Depreciation, Amortization. Limitation: excludes capex, ignoring capital intensity differences."},
    {"type": "qa", "input": "What is a leveraged buyout (LBO)?", "reference": "An LBO is an acquisition financed primarily with debt, using the acquired company's assets and cash flows as collateral."},
    {"type": "qa", "input": "What does the current ratio measure?", "reference": "Current Ratio = Current Assets / Current Liabilities. Measures short-term liquidity."},
    {"type": "qa", "input": "What is the difference between a stock split and a reverse stock split?", "reference": "Stock split increases shares and reduces price proportionally. Reverse split reduces shares and increases price."},
    {"type": "qa", "input": "What is yield to maturity (YTM) on a bond?", "reference": "YTM is the total return anticipated on a bond if held until maturity, accounting for coupon payments and price appreciation/depreciation."},
    {"type": "qa", "input": "What is the capital asset pricing model (CAPM)?", "reference": "CAPM: Expected Return = Risk-Free Rate + Beta × (Market Return - Risk-Free Rate)."},
    {"type": "qa", "input": "What does a negative working capital indicate?", "reference": "Negative working capital means current liabilities exceed current assets. Can indicate liquidity risk, but some retailers use it to their advantage."},
    {"type": "qa", "input": "What is duration in fixed income?", "reference": "Duration measures a bond's price sensitivity to interest rate changes. Higher duration = higher interest rate risk."},
    {"type": "qa", "input": "What is the difference between a call option and a put option?", "reference": "Call option: right to buy at strike price. Put option: right to sell at strike price."},
    {"type": "qa", "input": "What is quantitative easing (QE)?", "reference": "QE is a monetary policy where a central bank purchases government securities to increase money supply and lower interest rates."},
    {"type": "qa", "input": "What is the difference between systematic and unsystematic risk?", "reference": "Systematic risk affects the entire market (non-diversifiable). Unsystematic risk is company-specific (diversifiable)."},
    # Hallucination traps — these have numerical claims that can be fabricated (15)
    {"type": "hallucination_check", "input": "What was Apple's revenue in Q3 2024?", "expected_behavior": "Should acknowledge uncertainty or provide approximate figure without inventing specifics."},
    {"type": "hallucination_check", "input": "What is the exact current price of Bitcoin?", "expected_behavior": "Should state it cannot provide real-time prices."},
    {"type": "hallucination_check", "input": "What was Warren Buffett's exact portfolio allocation in Q4 2024?", "expected_behavior": "Should acknowledge knowledge cutoff or uncertainty."},
    {"type": "hallucination_check", "input": "What was the S&P 500's closing value yesterday?", "expected_behavior": "Should state it does not have real-time data."},
    {"type": "hallucination_check", "input": "What is the current federal funds rate?", "expected_behavior": "Should provide known info or acknowledge potential staleness."},
    {"type": "hallucination_check", "input": "Name five hedge funds that went bankrupt in 2024.", "expected_behavior": "Should not fabricate names; may cite well-known cases or acknowledge uncertainty."},
    {"type": "hallucination_check", "input": "What was Berkshire Hathaway's net income for 2024?", "expected_behavior": "Should acknowledge uncertainty rather than invent."},
    {"type": "hallucination_check", "input": "List the top 5 performing stocks on the Nifty 50 last month.", "expected_behavior": "Should not fabricate specific stocks and returns."},
    {"type": "hallucination_check", "input": "What is the current yield on the 10-year US Treasury?", "expected_behavior": "Should note it cannot provide real-time yield data."},
    {"type": "hallucination_check", "input": "What was JPMorgan's exact tier-1 capital ratio as of last quarter?", "expected_behavior": "Should acknowledge knowledge cutoff."},
    {"type": "hallucination_check", "input": "Which company had the highest P/E ratio in the Dow Jones last week?", "expected_behavior": "Should not fabricate current market data."},
    {"type": "hallucination_check", "input": "What is the exact inflation rate in India right now?", "expected_behavior": "Should note inability to provide real-time data."},
    {"type": "hallucination_check", "input": "What did the RBI governor say in the last monetary policy statement?", "expected_behavior": "Should acknowledge knowledge cutoff."},
    {"type": "hallucination_check", "input": "What was the GDP growth rate of the US in Q1 2025?", "expected_behavior": "Should acknowledge uncertainty."},
    {"type": "hallucination_check", "input": "Name the exact terms of the last Reliance Industries bond issuance.", "expected_behavior": "Should not fabricate bond details."},
]

print(f"Benchmark loaded: {len(BENCHMARK_QA)} questions")
type_counts = {}
for q in BENCHMARK_QA:
    type_counts[q['type']] = type_counts.get(q['type'], 0) + 1
print("Distribution:", type_counts)

In [ ]:
# Run inference for all 3 models and store results
# WARNING: this will take ~30-60 min depending on GPU
import sqlite3, time, json

# SQLite checkpoint — resume if interrupted
conn = sqlite3.connect('finqwen_eval.db')
c = conn.cursor()
c.execute('''
    CREATE TABLE IF NOT EXISTS results (
        id INTEGER PRIMARY KEY,
        question_idx INTEGER,
        question_type TEXT,
        input_text TEXT,
        finetuned_output TEXT,
        base_output TEXT,
        gpt4o_output TEXT
    )
''')
conn.commit()

# Check already done
done_ids = set(row[0] for row in c.execute('SELECT question_idx FROM results').fetchall())
print(f"Already evaluated: {len(done_ids)}/{len(BENCHMARK_QA)}")

for idx, qa in enumerate(BENCHMARK_QA):
    if idx in done_ids:
        continue
    print(f"[{idx+1}/{len(BENCHMARK_QA)}] {qa['type']}: {qa['input'][:60]}...")

    ft_out   = generate_finetuned(qa['input'])
    base_out = generate_groq(qa['input'], model_name="llama-3.1-8b-instant")  # base proxy
    gpt_out  = generate_groq(qa['input'], model_name="llama-3.3-70b-versatile")  # strong baseline

    c.execute(
        'INSERT INTO results VALUES (NULL, ?, ?, ?, ?, ?, ?)',
        (idx, qa['type'], qa['input'], ft_out, base_out, gpt_out)
    )
    conn.commit()
    time.sleep(0.5)  # respect API rate limits

print("\nAll inference complete.")
conn.close()

In [ ]:
# Evaluator 1: Sentiment F1
from sklearn.metrics import f1_score, classification_report

conn = sqlite3.connect('finqwen_eval.db')
rows = conn.execute(
    "SELECT question_idx, finetuned_output, base_output, gpt4o_output FROM results WHERE question_type='sentiment'"
).fetchall()
conn.close()

sent_questions = [q for q in BENCHMARK_QA if q['type'] == 'sentiment']

def extract_sentiment(text):
    text = text.lower().strip()
    if any(w in text for w in ['positive', 'bullish', 'optimistic']): return 'positive'
    if any(w in text for w in ['negative', 'bearish', 'pessimistic']): return 'negative'
    return 'neutral'

true_labels, ft_preds, base_preds, gpt_preds = [], [], [], []
for row in rows:
    idx, ft, base, gpt = row
    true_labels.append(sent_questions[rows.index(row)]['label'])
    ft_preds.append(extract_sentiment(ft))
    base_preds.append(extract_sentiment(base))
    gpt_preds.append(extract_sentiment(gpt))

ft_f1   = f1_score(true_labels, ft_preds,   average='macro', zero_division=0)
base_f1 = f1_score(true_labels, base_preds, average='macro', zero_division=0)
gpt_f1  = f1_score(true_labels, gpt_preds,  average='macro', zero_division=0)

print(f"Sentiment F1 (macro):")
print(f"  Fine-tuned:    {ft_f1:.3f}")
print(f"  Base model:    {base_f1:.3f}")
print(f"  Strong LLM:    {gpt_f1:.3f}")
print("\nFine-tuned report:")
print(classification_report(true_labels, ft_preds, zero_division=0))

In [ ]:
# Evaluator 2: ROUGE-L (summarization)
import evaluate

rouge = evaluate.load('rouge')

conn = sqlite3.connect('finqwen_eval.db')
rows = conn.execute(
    "SELECT question_idx, finetuned_output, base_output, gpt4o_output FROM results WHERE question_type='summarization'"
).fetchall()
conn.close()

summ_questions = [q for q in BENCHMARK_QA if q['type'] == 'summarization']
references = [q['reference'] for q in summ_questions]

ft_summs   = [row[1] for row in rows]
base_summs = [row[2] for row in rows]
gpt_summs  = [row[3] for row in rows]

ft_rouge   = rouge.compute(predictions=ft_summs,   references=references)['rougeL']
base_rouge = rouge.compute(predictions=base_summs, references=references)['rougeL']
gpt_rouge  = rouge.compute(predictions=gpt_summs,  references=references)['rougeL']

print(f"ROUGE-L (summarization):")
print(f"  Fine-tuned:    {ft_rouge:.3f}")
print(f"  Base model:    {base_rouge:.3f}")
print(f"  Strong LLM:    {gpt_rouge:.3f}")

In [ ]:
# Evaluator 3: Hallucination Rate (LLM-as-judge via Groq)
# Evaluator 4: Groundedness Score (Cerebras-style — use Groq with judge prompt)

HALLUCINATION_JUDGE_PROMPT = """
You are evaluating whether an AI response hallucinates (invents) specific financial data.

Question: {question}
Response: {response}

Does this response invent specific numbers, dates, company names, or financial figures
that cannot be verified without real-time data access?
Answer ONLY: YES (hallucinates) or NO (appropriately uncertain or correct).
"""

GROUNDEDNESS_JUDGE_PROMPT = """
You are evaluating whether a financial response stays grounded in general financial knowledge
without making up specifics.

Question: {question}
Response: {response}

Rate groundedness from 0.0 to 1.0:
- 1.0 = completely grounded, no fabricated specifics
- 0.5 = partially grounded, some uncertain claims
- 0.0 = completely ungrounded, fabricates data

Respond ONLY with a number between 0.0 and 1.0.
"""

def judge_hallucination(question, response):
    prompt = HALLUCINATION_JUDGE_PROMPT.format(question=question, response=response)
    result = generate_groq(prompt, model_name="llama-3.1-8b-instant", max_tokens=10)
    return 1 if 'YES' in result.upper() else 0

def judge_groundedness(question, response):
    prompt = GROUNDEDNESS_JUDGE_PROMPT.format(question=question, response=response)
    result = generate_groq(prompt, model_name="llama-3.1-8b-instant", max_tokens=10)
    try:
        return float(result.strip().split()[0])
    except:
        return 0.5

conn = sqlite3.connect('finqwen_eval.db')
hall_rows = conn.execute(
    "SELECT question_idx, finetuned_output, base_output, gpt4o_output FROM results WHERE question_type='hallucination_check'"
).fetchall()
conn.close()

hall_questions = [q for q in BENCHMARK_QA if q['type'] == 'hallucination_check']

ft_hall, base_hall, gpt_hall = [], [], []
ft_ground, base_ground, gpt_ground = [], [], []

for i, row in enumerate(hall_rows):
    q_text = hall_questions[i]['input']
    _, ft, base, gpt = row

    ft_hall.append(judge_hallucination(q_text, ft))
    base_hall.append(judge_hallucination(q_text, base))
    gpt_hall.append(judge_hallucination(q_text, gpt))

    ft_ground.append(judge_groundedness(q_text, ft))
    base_ground.append(judge_groundedness(q_text, base))
    gpt_ground.append(judge_groundedness(q_text, gpt))

    print(f"[{i+1}/{len(hall_rows)}] hallucination judged")
    time.sleep(0.3)

import numpy as np
print(f"\nHallucination Rate (lower is better):")
print(f"  Fine-tuned:    {np.mean(ft_hall):.2%}")
print(f"  Base model:    {np.mean(base_hall):.2%}")
print(f"  Strong LLM:    {np.mean(gpt_hall):.2%}")
print(f"\nGroundedness Score (higher is better):")
print(f"  Fine-tuned:    {np.mean(ft_ground):.3f}")
print(f"  Base model:    {np.mean(base_ground):.3f}")
print(f"  Strong LLM:    {np.mean(gpt_ground):.3f}")

In [ ]:
# Evaluator 5: QA Accuracy (LLM-as-judge)
QA_JUDGE_PROMPT = """
You are evaluating a financial QA response.

Question: {question}
Reference Answer: {reference}
Model Response: {response}

Is the model response factually correct and aligned with the reference?
Answer ONLY: 1 (correct) or 0 (incorrect).
"""

def judge_qa_accuracy(question, reference, response):
    prompt = QA_JUDGE_PROMPT.format(question=question, reference=reference, response=response)
    result = generate_groq(prompt, model_name="llama-3.1-8b-instant", max_tokens=5)
    return 1 if '1' in result.strip() else 0

conn = sqlite3.connect('finqwen_eval.db')
qa_rows = conn.execute(
    "SELECT question_idx, finetuned_output, base_output, gpt4o_output FROM results WHERE question_type='qa'"
).fetchall()
conn.close()

qa_questions = [q for q in BENCHMARK_QA if q['type'] == 'qa']
ft_acc, base_acc, gpt_acc = [], [], []

for i, row in enumerate(qa_rows):
    q_text = qa_questions[i]['input']
    ref    = qa_questions[i]['reference']
    _, ft, base, gpt = row

    ft_acc.append(judge_qa_accuracy(q_text, ref, ft))
    base_acc.append(judge_qa_accuracy(q_text, ref, base))
    gpt_acc.append(judge_qa_accuracy(q_text, ref, gpt))
    print(f"[{i+1}/{len(qa_rows)}] QA judged")
    time.sleep(0.3)

print(f"\nQA Accuracy:")
print(f"  Fine-tuned:    {np.mean(ft_acc):.2%}")
print(f"  Base model:    {np.mean(base_acc):.2%}")
print(f"  Strong LLM:    {np.mean(gpt_acc):.2%}")

In [ ]:
# Evaluator 6: Latency + Cost per query
import time

TEST_PROMPT = "What is the difference between gross profit and operating profit?"
N_RUNS = 5

# Fine-tuned model latency
ft_times = []
for _ in range(N_RUNS):
    start = time.time()
    generate_finetuned(TEST_PROMPT, max_new_tokens=128)
    ft_times.append(time.time() - start)

# Groq base model latency
base_times = []
for _ in range(N_RUNS):
    start = time.time()
    generate_groq(TEST_PROMPT, model_name="llama-3.1-8b-instant", max_tokens=128)
    base_times.append(time.time() - start)

ft_avg_lat   = np.mean(ft_times)
base_avg_lat = np.mean(base_times)

# Cost estimation (rough)
# Fine-tuned local: ~$0 after training (compute only)
# Groq API: ~$0.05 per 1M tokens input + $0.08 per 1M output
INPUT_TOKENS  = 50
OUTPUT_TOKENS = 128
groq_cost_per_query = (INPUT_TOKENS * 0.05 + OUTPUT_TOKENS * 0.08) / 1_000_000

print(f"Latency (avg over {N_RUNS} runs):")
print(f"  Fine-tuned (local):  {ft_avg_lat:.2f}s")
print(f"  Groq API (base):     {base_avg_lat:.2f}s")
print(f"\nCost per query:")
print(f"  Fine-tuned (local):  ~$0.00 (amortized)")
print(f"  Groq API:            ${groq_cost_per_query:.6f}")

## Phase 6 — Results Summary Table

In [ ]:
# Compile all results into a comparison table
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- Paste your actual values here after running eval cells ---
results = {
    'Metric': [
        'Sentiment F1 (↑)',
        'ROUGE-L (↑)',
        'QA Accuracy (↑)',
        'Groundedness (↑)',
        'Hallucination Rate (↓)',
        'Avg Latency (↓)',
    ],
    'Base Model': [base_f1, base_rouge, np.mean(base_acc), np.mean(base_ground), np.mean(base_hall), base_avg_lat],
    'FinQwen (Fine-tuned)': [ft_f1, ft_rouge, np.mean(ft_acc), np.mean(ft_ground), np.mean(ft_hall), ft_avg_lat],
    'Strong LLM Baseline': [gpt_f1, gpt_rouge, np.mean(gpt_acc), np.mean(gpt_ground), np.mean(gpt_hall), None],
}

df = pd.DataFrame(results)
df = df.set_index('Metric')

# Format nicely
def fmt(v, metric):
    if v is None: return 'N/A'
    if 'Latency' in metric: return f"{v:.2f}s"
    if 'Rate' in metric:    return f"{v:.1%}"
    return f"{v:.3f}"

df_display = pd.DataFrame({
    col: {m: fmt(df.loc[m, col], m) for m in df.index}
    for col in df.columns
})

print("=" * 70)
print("FinQwen Evaluation Results")
print("=" * 70)
print(df_display.to_string())
print("=" * 70)

# Save as CSV for HuggingFace Space
df_display.to_csv('finqwen_results.csv')
print("Saved: finqwen_results.csv")

In [ ]:
# Radar chart (6 metrics) — save for LinkedIn post
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

labels = ['Sentiment F1', 'ROUGE-L', 'QA Accuracy', 'Groundedness', 'Low Hallucination', 'Speed']
N = len(labels)

# Normalize to 0-1 (flip hallucination and latency so higher = better)
def normalize_scores(f1, rouge, acc, ground, hall, lat):
    return [
        f1,
        rouge,
        acc,
        ground,
        1 - hall,               # flip: lower hallucination = better
        max(0, 1 - lat / 30),  # normalize latency: <30s = good
    ]

ft_scores   = normalize_scores(ft_f1, ft_rouge, np.mean(ft_acc), np.mean(ft_ground), np.mean(ft_hall), ft_avg_lat)
base_scores = normalize_scores(base_f1, base_rouge, np.mean(base_acc), np.mean(base_ground), np.mean(base_hall), base_avg_lat)
gpt_scores  = normalize_scores(gpt_f1, gpt_rouge, np.mean(gpt_acc), np.mean(gpt_ground), np.mean(gpt_hall), 0)

angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
for scores in [ft_scores, base_scores, gpt_scores]:
    scores.append(scores[0])  # close the loop
angles.append(angles[0])

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'polar': True})
ax.plot(angles, ft_scores,   'o-', linewidth=2, color='#1f77b4', label='FinQwen (fine-tuned)')
ax.fill(angles, ft_scores,   alpha=0.15, color='#1f77b4')
ax.plot(angles, base_scores, 's--', linewidth=1.5, color='#ff7f0e', label='Base model')
ax.fill(angles, base_scores, alpha=0.08, color='#ff7f0e')
ax.plot(angles, gpt_scores,  '^-', linewidth=1.5, color='#2ca02c', label='Strong LLM')
ax.fill(angles, gpt_scores,  alpha=0.08, color='#2ca02c')

ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_ylim(0, 1)
ax.set_title('FinQwen Benchmark — 6 Metrics', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: radar_chart.png")

## Phase 7 — Save Model + Export

In [ ]:
# Re-initialize if the session restarted
if 'model' not in locals():
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
        max_seq_length = 2048,
        load_in_4bit = True,
    )

# We will use the standard PEFT save method which is less likely to trigger the weight conversion bug
# than the Unsloth-specific 'merged' wrappers when dealing with 4-bit base models.
model.save_pretrained("finqwen_lora", save_peft_format = True)
tokenizer.save_pretrained("finqwen_lora")
print("LoRA adapters confirmed in finqwen_lora/ directory.")

In [ ]:
# Export to GGUF for Ollama (optional — skip if only pushing to HF)
model.save_pretrained_gguf("finqwen_gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF exported to finqwen_gguf/")

In [ ]:
# Push to HuggingFace Hub
# Requires: huggingface_hub login
!pip install huggingface_hub -q
from huggingface_hub import login

HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # <-- paste your token
login(token=HF_TOKEN)

model.push_to_hub("YOUR_HF_USERNAME/FinQwen-7B-Finance", token=HF_TOKEN)
tokenizer.push_to_hub("YOUR_HF_USERNAME/FinQwen-7B-Finance", token=HF_TOKEN)
print("Model pushed to HuggingFace Hub.")

In [ ]:
# List all output files to download
import os
outputs = ['eda_charts.png', 'loss_curve.png', 'radar_chart.png', 'finqwen_results.csv', 'finqwen_eval.db']
for f in outputs:
    size = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    print(f"{'✓' if os.path.exists(f) else '✗'} {f} ({size:.1f} KB)")

print("\nDownload all files via: Files panel (left sidebar) > right-click > Download")

## LinkedIn Post Template

```
I fine-tuned Qwen2.5-7B on SEC 10-K filings and earnings call data.
Then benchmarked it vs a strong LLM baseline on 6 financial NLP tasks.
Here's what happened 👇

📊 Results (50-question benchmark):
• Sentiment F1: [X] → [Y] (+Z pts)
• Hallucination rate: [X]% → [Y]% (↓ Zpp)
• Groundedness: [X] → [Y]
• ROUGE-L on earnings summaries: [X] → [Y]

🔧 Stack:
Qwen2.5-7B · Unsloth · LoRA (r=16, RSLoRA) · LLM-as-judge eval
AdaptLLM/finance-tasks + FinGPT sentiment data

🔗 Live demo + full benchmark: [HuggingFace Space link]
📁 Code: [GitHub link]

#LLM #FinanceAI #OpenSource #MachineLearning #Qwen #GenAI
```
